In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import os
import sys
import seaborn as sns
import glob
import torch
import scipy.stats as stats
import scipy
sys.path.append('/n/groups/marks/projects/viral_families/models/SaProt/')
from utils.foldseek_util import get_struc_seq
os.chdir('/n/groups/marks/projects/viral_families/models/SaProt/')

In [2]:
ref = pd.read_csv("/n/groups/marks/projects/viral_families/priority-viruses/data/reference_files/viral_dms_reference.csv")

In [3]:
ref

,DMS ID,Viral Family,Virus,Protein,Author,Title,Year,Assay,Type,In ProteinGym,Sequence
0,LASSA_GP_Carr,Arenaviridae,Lassa,GP,Carr,Deep mutational scanning reveals functional co...,2024,fitness,Eukaryotic virus,No,MGQIVTFFQEVPHVIEEVMNIVLIALSVLAVLKGLYNFATCGLVGL...
1,PESV_POLG_Tsuboyama,Caliciviridae,Porcine enteric sapovirus,POLG,Tsuboyama,Mega-scale experimental analysis of protein fo...,2023,stability,Eukaryotic virus,Yes,ALRDDEYDEWQDIIRDWRKEMTVQQFLDLKERALSGASDPDSQRYN...
2,SARS2_PLPRO_Wu_abundance,Coronaviridae,SARS-CoV-2,PLPRO,Wu,Mutational profiling of SARS-CoV-2 papain-like...,2024,abundance,Eukaryotic virus,No,MEVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIK...
3,SARS2_PLPRO_Wu_activity,Coronaviridae,SARS-CoV-2,PLPRO,Wu,Mutational profiling of SARS-CoV-2 papain-like...,2024,activity,Eukaryotic virus,No,MEVRTIKVFTTVDNINLHTQVVDMSMTYGQQFGPTYLDGADVTKIK...
4,SARS2_PRD0038_RBD_Starr,Coronaviridae,Bat coronavirus PRD0038,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MKFFILLSLLPFATAQEGCGILSNKSKPALTQYSSSRRGFYYFDDT...
5,SARS2_MRPO_Flynn,Coronaviridae,SARS-CoV-2,MRPO,Flynn,Comprehensive fitness landscape of SARS-CoV-2 ...,2022,fitness,Eukaryotic virus,Yes,SGFRKMAFPSGKVEGCMVQVTCGTTTLNGLWLDDVVYCPRHVICTS...
6,RmYN02_RBD_Starr,Coronaviridae,Bat coronavirus RmYN02,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MFILLLIGYTAATTCVTGPTTENKQNVSSLMRGVYYPDDIYRSNVN...
7,RsYN04_RBD_Starr,Coronaviridae,Bat coronavirus RsYN04,RBD,Starr,https://github.com/tstarrlab/SARSr-CoV-RBD_DMS,2023,expression,Eukaryotic virus,No,MFILLLLPIVLAQQDSCNHIVQLPNSMVRGVYNSGSKVYYPDDINR...
8,SARS2_XBB15_RBD_Taylor,Coronaviridae,SARS-CoV-2 XBB.1.5,RBD,Taylor,Deep mutational scans of XBB.1.5 and BQ.1.1 re...,2023,expression,Eukaryotic virus,No,MFVFLVLLPLVSSQCVNLITRTQSYTNSFTRGVYYPDKVFRSSVLH...
9,SARS2_BA1_SPIKE_Dadonaite,Coronaviridae,SARS-CoV-2 BA.1,SPIKE,Dadonaite,A pseudovirus system enables deep mutational s...,2023,fitness,Eukaryotic virus,No,MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSS...


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, EsmTokenizer, EsmForMaskedLM

# ---------------- CONFIG ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHAIN = ["A"]
MAX_LEN = 1024

ID_SERIES = ref["DMS ID"]          # or: data["DMS ID"]

PDB_DIR = "/n/groups/marks/projects/viral_families/priority-viruses/data/viral_dms_structures"
FOLDSEEK_BIN = "bin/foldseek"

SAPROT_NAME = "westlake-repl/SaProt_650M_PDB"
ESM_NAME    = "facebook/esm1v_t33_650M_UR90S_1"
# ----------------------------------------


def compute_pppl_masked_lm(model, tokenizer, seq: str, device=DEVICE) -> float:
    """
    Pseudo-perplexity via leave-one-out masking:
      pppl = exp( - (1/L) * sum_i log p(x_i | x_{-i}) )
    """
    model.eval()
    token_ids = tokenizer.encode(seq, return_tensors="pt").to(device)
    L = token_ids.size(1)

    log_likelihood = 0.0
    for i in range(L):
        masked = token_ids.clone()
        masked[0, i] = tokenizer.mask_token_id

        with torch.no_grad():
            out = model(masked)
            logp = torch.nn.functional.log_softmax(out.logits, dim=-1)

        tok = token_ids[0, i].item()
        if tok < 0 or tok >= logp.size(-1):
            return np.nan
        log_likelihood += logp[0, i, tok].item()

    avg_ll = log_likelihood / L
    return float(np.exp(-avg_ll))


# -------- Load both models/tokenizers once --------
saprot_tok = EsmTokenizer.from_pretrained(SAPROT_NAME)
saprot_model = EsmForMaskedLM.from_pretrained(SAPROT_NAME).to(DEVICE)

esm_tok = AutoTokenizer.from_pretrained(ESM_NAME)
esm_model = EsmForMaskedLM.from_pretrained(ESM_NAME).to(DEVICE)


# -------- Single pass loop, collect rows for BOTH tables --------
rows_saprot = []
rows_esm = []

for name in ID_SERIES.values:
    pdb_path = os.path.join(PDB_DIR, f"{name}.pdb")

    # shared status bookkeeping
    status = "ok"
    seq_len = np.nan

    if not os.path.exists(pdb_path):
        status = "missing_pdb"
        rows_saprot.append({"DMS_ID": name, "pdb_path": pdb_path, "seq_len": seq_len,
                            "pppl": np.nan, "status": status})
        rows_esm.append({"DMS_ID": name, "pdb_path": pdb_path, "seq_len": seq_len,
                         "pppl": np.nan, "status": status})
        continue

    try:
        parsed = get_struc_seq(FOLDSEEK_BIN, pdb_path, CHAIN)["A"]
        seq, foldseek_seq, combined_seq = parsed
        seq_len = len(seq)
    except Exception as e:
        status = f"foldseek_fail:{type(e).__name__}"
        rows_saprot.append({"DMS_ID": name, "pdb_path": pdb_path, "seq_len": seq_len,
                            "pppl": np.nan, "status": status})
        rows_esm.append({"DMS_ID": name, "pdb_path": pdb_path, "seq_len": seq_len,
                         "pppl": np.nan, "status": status})
        continue

    if seq_len > MAX_LEN:
        status = "too_long"
        rows_saprot.append({"DMS_ID": name, "pdb_path": pdb_path, "seq_len": seq_len,
                            "pppl": np.nan, "status": status})
        rows_esm.append({"DMS_ID": name, "pdb_path": pdb_path, "seq_len": seq_len,
                         "pppl": np.nan, "status": status})
        continue

    # --- SaProt pppl ---
    try:
        pppl_saprot = compute_pppl_masked_lm(saprot_model, saprot_tok, seq, DEVICE)
    except Exception as e:
        pppl_saprot = np.nan
        status_saprot = f"saprot_fail:{type(e).__name__}"
    else:
        status_saprot = status

    rows_saprot.append({
        "DMS_ID": name,
        "pdb_path": pdb_path,
        "seq_len": seq_len,
        "pppl": pppl_saprot,
        "status": status_saprot,
    })

    # --- ESM1v pppl ---
    try:
        pppl_esm = compute_pppl_masked_lm(esm_model, esm_tok, seq, DEVICE)
    except Exception as e:
        pppl_esm = np.nan
        status_esm = f"esm_fail:{type(e).__name__}"
    else:
        status_esm = status

    rows_esm.append({
        "DMS_ID": name,
        "pdb_path": pdb_path,
        "seq_len": seq_len,
        "pppl": pppl_esm,
        "status": status_esm,
    })

    print(f"{name} | SaProt pppl={pppl_saprot} | ESM1v pppl={pppl_esm}")


# -------- Two separate tables --------
pppl_saprot_df = pd.DataFrame(rows_saprot).sort_values(["status", "DMS_ID"])
pppl_esm_df    = pd.DataFrame(rows_esm).sort_values(["status", "DMS_ID"])

#pppl_saprot_df, pppl_esm_df

Some weights of EsmForMaskedLM were not initialized from the model checkpoint at westlake-repl/SaProt_650M_PDB and are newly initialized: ['esm.contact_head.regression.weight', 'esm.embeddings.position_embeddings.weight', 'esm.contact_head.regression.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of EsmForMaskedLM were not initialized from the model checkpoint at facebook/esm1v_t33_650M_UR90S_1 and are newly initialized: ['esm.contact_head.regression.weight', 'esm.contact_head.regression.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
import numpy as np
import pandas as pd

# ---------------- CONFIG ----------------
SEQ_COL = "Sequence"    # in ref
ID_COL  = "DMS ID"      # in ref
# ---------------------------------------


def make_pppl_tranception_table(
    ref: pd.DataFrame,
    model,
    *,
    id_col: str = ID_COL,
    seq_col: str = SEQ_COL,
    batch_size_inference: int = 8,
    num_workers: int = 1,
    scoring_mirror: bool = True,
    indel_mode: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      1) pppl_tranception_df: minimal table keyed by mutated_sequence
      2) ref_with_pppl: ref merged with scores + pppl_tranception
    """

    # ---- 1) Start clean ----
    df_tranception = ref.copy()

    # Basic sanity: keep rows with sequences
    df_tranception = df_tranception.dropna(subset=[seq_col]).copy()
    df_tranception[seq_col] = df_tranception[seq_col].astype(str)

    # ---- 2) Build DMS_data table expected by Tranception ----
    # Tranception expects: mutant, mutated_sequence, DMS_score, DMS_score_bin
    seq_df = pd.DataFrame({
        "mutant": df_tranception[id_col].astype(str).fillna(""),
        "mutated_sequence": df_tranception[seq_col],
        "DMS_score": [[0]] * len(df_tranception),
        "DMS_score_bin": [[0]] * len(df_tranception),
    })

    # Optional: drop duplicate sequences to avoid redundant scoring
    # (keeps the first occurrence; adjust if you prefer otherwise)
    seq_df = seq_df.drop_duplicates(subset=["mutated_sequence"]).reset_index(drop=True)

    # ---- 3) Score ----
    scores = model.score_mutants(
        DMS_data=seq_df,
        target_seq=None,
        scoring_mirror=scoring_mirror,
        batch_size_inference=batch_size_inference,
        num_workers=num_workers,
        indel_mode=indel_mode,
    )

    # ---- 4) Compute pppl ----
    # Your earlier convention: pppl = exp(-avg_score)
    if "avg_score" not in scores.columns:
        raise ValueError("Expected `scores` to contain an `avg_score` column.")

    scores = scores.copy()
    scores["pppl_tranception"] = np.exp(-scores["avg_score"].astype(float))

    # Keep a neat table for downstream merges/plots
    pppl_tranception_df = scores[["mutated_sequence", "avg_score", "pppl_tranception"]].copy()

    # ---- 5) Merge back to original ref (by sequence) ----
    ref_with_pppl = ref.copy()
    ref_with_pppl = ref_with_pppl.merge(
        pppl_tranception_df,
        left_on=seq_col,
        right_on="mutated_sequence",
        how="left",
    )

    return pppl_tranception_df, ref_with_pppl


# ------------------- USAGE -------------------
pppl_tranception_df, ref_with_pppl_tranception = make_pppl_tranception_table(
    ref,
    model,
    batch_size_inference=8,  # bump this up to speed up
    num_workers=1,
    scoring_mirror=True,
    indel_mode=False,
)

# pppl_tranception_df is your "new datatable"
pppl_tranception_df